<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINT-Colab/blob/main/notebooks/01_decisao_qwen35_osint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 01 · Modelo de decisão OSINT com Qwen3.5 4B + LoRA

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINT-Colab)

Reconstrução do notebook *Qwen3.5 (4B) Decision* da Unsloth aplicada a OSINT: triagem de perfis imitadores com **probabilidade calibrada** e, o que o original não faz, **auditoria do gabarito** (teto do professor e lacuna humana) antes de confiar no número.

| | |
|---|---|
| **Hardware** | GPU T4 gratuita (Ambiente de execução → Alterar tipo → T4 GPU) |
| **Tempo estimado** | 15 a 40 min no T4 com o dataset de demonstração (o original leva ~2h com o typed-decisions completo) |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

### Como funciona (resumo da engenharia reversa)
1. `FastDecisionModel` carrega um LLM normal (Qwen3.5 4B em 4 bits) e **acopla uma cabeça de decisão** pequena que lê todas as perguntas sobre uma entrada em uma passada e pontua cada opção.
2. LoRA ajusta poucas matrizes do LLM; a cabeça nova sempre treina.
3. O treino usa gold **suave** (probabilidades), não só o rótulo vencedor.
4. `calibrate` ajusta as probabilidades no holdout (ECE menor = confiança honesta).
5. **OSINTIA:** separamos antes um conjunto de auditoria verificado por humano, que nunca entra no treino.

In [ ]:
%%capture
# Instalação igual à dos notebooks oficiais da Unsloth (LGPL-3.0), mantida para compatibilidade.
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINT-Colab.git /content/OSINT-Colab 2>/dev/null || git -C /content/OSINT-Colab pull -q
import sys
sys.path.insert(0, "/content/OSINT-Colab")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

## 1. Carregar o modelo
Troque `MODELO` por `unsloth/Llama-3.2-3B-Instruct` ou outro suportado se quiser comparar.

In [ ]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch

MODELO = "unsloth/Qwen3.5-4B"
model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = MODELO,
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastDecisionModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 2. Dados
`FONTE` escolhe o que treinar:
- `"demo"`: só a triagem de perfis OSINT (rápido)
- `"demo+seguranca"`: soma o subconjunto `security_incidents` do typed-decisions (mais dados, mais tempo)

Primeiro separamos **80 perfis de auditoria** com rótulo humano. Eles nunca entram no treino nem na calibração.

In [ ]:
import json
FONTE = "demo"  # ou "demo+seguranca"

linhas = dados.carregar_jsonl("/content/OSINT-Colab/dados/demo/triagem_perfis.jsonl")
treino_linhas, auditoria_linhas = dados.dividir_holdout(linhas, fracao=0.2, semente=3407)
print(len(treino_linhas), "para treino/calibração |", len(auditoria_linhas), "para auditoria humana")

linhas_treino = [{"state": l["state"], "questions": l["questions"], "gold": l["gold"]} for l in treino_linhas]

if FONTE == "demo+seguranca":
    from datasets import load_dataset
    seg = load_dataset("LocalLLaMA/typed-decisions", "security_incidents", split="train")
    linhas_treino += [{"state": r["state"], "questions": r["questions"], "gold": r["gold"]} for r in seg]

print(len(linhas_treino), "linhas de treino")

Conferindo o formato: o gold oficial do typed-decisions e o nosso têm a mesma estrutura (`label`, `probabilities`, `confidence`, `type`).

In [ ]:
print(json.dumps(json.loads(linhas_treino[0]["gold"]), indent=2, ensure_ascii=False))

In [ ]:
items, report = FastDecisionModel.build_dataset(linhas_treino, tokenizer, model)
print(f"Descartadas {report['skipped']} de {report['total']} decisões")
train_items, eval_items = FastDecisionModel.split_holdout(items, seed = 3407)
print(f"{len(train_items)} para treino, {len(eval_items)} para calibração")
FastDecisionModel.evaluate(model, tokenizer, eval_items)  # antes do treino: perto do chute

## 3. Treinar

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = eval_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,      # dataset pequeno: épocas em vez de max_steps
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 5,
        output_dir = "outputs",
        report_to = "none",
        seed = 3407,
    ),
)
trainer_stats = trainer.train()

## 4. Calibrar no holdout

In [ ]:
FastDecisionModel.calibrate(model, tokenizer, eval_items)
FastDecisionModel.for_inference(model)

## 5. Auditoria OSINTIA: o modelo aprendeu a tarefa ou o professor?
Rodamos o modelo nos 80 perfis de auditoria e comparamos com o **gold** (professor) e com o **humano**.

In [ ]:
perguntas = esquemas.TRIAGEM_PERFIL
prev, probs = [], []
for l in auditoria_linhas:
    resp = FastDecisionModel.predict(model, tokenizer, l["state"], perguntas)
    prev.append(resp["perfil"]["answer"])
    probs.append(resp["perfil"]["probabilities"])

gold_aud   = dados.rotulos(auditoria_linhas, "perfil")
humano_aud = [l["rotulo_humano"]["perfil"] for l in auditoria_linhas]
extra_aud  = [l["amostra_professor_extra"]["perfil"] for l in auditoria_linhas]

teto = auditoria.teto_do_professor([extra_aud], gold_aud)
print("vs gold:  ", calibracao.relatorio(probs, gold_aud))
print("vs humano:", calibracao.relatorio(probs, humano_aud))
print(f"teto do professor: {teto:.3f}")
print(auditoria.lacuna_humana(prev, gold_aud, humano_aud))
acc_gold = auditoria.concordancia(prev, gold_aud)
if auditoria.alerta_de_decoracao(acc_gold, teto):
    print("⚠️  O modelo concorda com o gold mais do que o próprio professor: está decorando os vícios dele.")

In [ ]:
herdados = auditoria.erros_herdados(prev, gold_aud, humano_aud)
print(f"{len(herdados)} erros herdados do professor (a métrica oficial conta como acerto)")
for i in herdados[:3]:
    print("-" * 60)
    print(auditoria_linhas[i]["state"])
    print("modelo:", prev[i], {k: round(v, 3) for k, v in probs[i].items()}, "| humano:", humano_aud[i])

## 6. Escolher o limiar de revisão humana

In [ ]:
for linha in roteamento.varrer_limiares(probs, humano_aud):
    print(linha)

## 7. Decidir um perfil novo
As perguntas podem misturar `choice`, `noul` e `score`. A pergunta `pede_dados` não estava no treino: a cabeça de decisão lê a instrução, mas a qualidade em tipos não treinados é menor. Para produção, inclua exemplos rotulados de cada pergunta que vai usar.

In [ ]:
novo = """Alvo monitorado: @bancoaurora
Perfil analisado: @bancoaurora.suporte24h
Selo de verificado: sim
Conta criada há 6 dias
Seguidores: 312
Média de postagens por dia: 9.5
Bio: Central de estorno. Mande seu CPF e o código recebido por SMS no direct.
Link na bio: https://aurora-estorno.example
Conteúdo repetido entre postagens: sim"""

perguntas_novas = dict(esquemas.TRIAGEM_PERFIL)
perguntas_novas["pede_dados"] = {"type": "noul", "instructions": "O perfil pede dados pessoais, senha ou código?"}

resp = FastDecisionModel.predict(model, tokenizer, novo, perguntas_novas)
for nome, r in resp.items():
    print(nome, r["answer"], {k: round(v, 3) for k, v in r["probabilities"].items()})
print("encaminhamento:", roteamento.rotear(resp["perfil"]["probabilities"], limiar=0.75))

## 8. Registrar e salvar

In [ ]:
import os
os.makedirs("/content/saidas", exist_ok=True)
dados.salvar_jsonl(
    [{"id": l["id"], "previsto": p, "probabilidades": pr, "humano": h} for l, p, pr, h in zip(auditoria_linhas, prev, probs, humano_aud)],
    "/content/saidas/auditoria_decisao.jsonl",
)
custodia.registrar("/content/saidas/manifesto.jsonl", "auditoria_modelo_decisao",
                   "/content/saidas/auditoria_decisao.jsonl", {"modelo": MODELO, "teto": round(teto, 4)})

model.save_pretrained("osintia_decisao_lora")  # LoRA + cabeça de decisão + calibração
# model.push_to_hub("seu_usuario/osintia_decisao_lora", token = "SEU_TOKEN_HF")

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*